# EX_02 — Embeddings con Transformers (ejercicios)

**Notebook de referencia:** `notebook/02_Embeddings_Transformers.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Mean pooling

Con `AutoTokenizer` + `AutoModel`, obtén **last_hidden_state** para una frase y calcula el embedding de frase como media sobre tokens (excluyendo padding).


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

text = "Transformers build contextual embeddings."

# TODO: tokenize, forward, mean pool (ignore pad)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")
inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)
    last_hidden_state = outputs.last_hidden_state
embeddings = outputs.last_hidden_state
# Mean pool (ignore pad tokens)
pooled_embeddings = embeddings.mean(dim=1)

print(pooled_embeddings.shape)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1474.80it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 768])


## Actividad 2 — `sentence-transformers`

Usa `SentenceTransformer` para embedder dos frases y calcula similitud coseno. Comenta brevemente (en inglés en un comentario) por qué suele ser mejor que mean-pooling manual de BERT base.


In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

# TODO: encode two sentences, cosine similarity
sentences = ["This is the first sentence.", "This is the second sentence."]
model = SentenceTransformer('all-MiniLM-L6-v2')
sentence_embeddings = model.encode(sentences)
cosine_sim = util.cos_sim(sentence_embeddings[0], sentence_embeddings[1])
print(sentence_embeddings.shape)
print(f"Cosine similarity: {cosine_sim}")

# ==============================================================================
# WHY SENTENCE-TRANSFORMERS IS BETTER THAN MANUAL MEAN-POOLING ON BERT BASE?
# ==============================================================================
"""
# Standard BERT-base is trained for Next Sentence Prediction and Masked Language Modeling. 
# Out of the box, its raw token embeddings suffer from anisotropy (they occupy a narrow cone 
# in the vector space), meaning even unrelated sentences get high cosine similarity (e.g., 0.6 - 0.8).

# SentenceTransformers (Bi-Encoders like SBERT) are explicitly fine-tuned using Siamese networks 
# (e.g., Contrastive Loss or Multiple Negatives Ranking Loss). This process maps sentences into 
# a structured vector space where semantic closeness directly correlates with cosine distance. 
# It yields significantly better performance on semantic textual similarity (STS) tasks 
# while requiring zero manual pooling code.
"""


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7835.98it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(2, 384)
Cosine similarity: tensor([[0.8953]])


## Actividad 3 — Paráfrasis

Escribe dos paráfrasis de una misma idea y muestra que sus embeddings (sentence-transformers) tienen **mayor** similitud entre sí que con una frase de tema distinto.


In [8]:
# TODO: three strings, 2 paraphrase + 1 unrelated, print cosine sims

model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "AI is changing the way we work.",
    "The way people work is being transformed by AI.",
    "I like to cook pasta for dinner."
]

emb = model.encode(sentences)

sim_paraphrase = util.cos_sim(emb[0], emb[1])
sim_distinta = util.cos_sim(emb[0], emb[2])

print(f"Similitud paráfrasis: {sim_paraphrase.item():.4f}")
print(f"Similitud distinta:   {sim_distinta.item():.4f}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3869.60it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Similitud paráfrasis: 0.8336
Similitud distinta:   0.1358
